# RBE 577 – Project 4: Imitation Learning Expert

This notebook walks through the **complete pipeline** for Project 4:
1. Environment setup
2. Collecting expert demonstrations
3. Converting data to robomimic format
4. Training Behavioral Cloning (BC)
5. Training Diffusion Policy
6. Evaluation & visualization
7. Discussion analysis & plots

---
> **Configuration** – Change the variables in the `CONFIGURATION` cell below to match your setup before running anything else.

## 0. CONFIGURATION – Edit These First

In [ ]:
# ============================================================
#  PROJECT CONFIGURATION  –  edit all paths / choices here
# ============================================================

# --- Task & robot -------------------------------------------------
TASK_NAME   = "PickPlaceCan"        # PickPlaceCan | Stack | Wipe
ROBOT_NAME  = "Panda"               # Panda | Sawyer | Kinova3 | UR5e

# --- Paths --------------------------------------------------------
BASE_DIR            = "/tmp/rbe577_project4"          # root output folder
DEMO_DIR            = f"{BASE_DIR}/demos"             # raw demo output
DEMO_HDF5           = f"{DEMO_DIR}/demo.hdf5"         # raw demo file
OBS_HDF5            = f"{BASE_DIR}/obs_dataset.hdf5"  # image-obs dataset
ROBOMIMIC_ROOT      = "/opt/robomimic"                # path to robomimic repo
ROBOSUITE_ROOT      = "/opt/robosuite"                # path to robosuite repo
BC_OUTPUT_DIR       = f"{BASE_DIR}/bc_output"
DIFF_OUTPUT_DIR     = f"{BASE_DIR}/diffusion_output"

# --- Data collection ----------------------------------------------
MIN_DEMOS           = 50    # must collect at least this many
TRAIN_VAL_RATIO     = 0.2   # fraction of data for validation

# --- Camera / rendering -------------------------------------------
CAMERA_NAMES        = ["agentview", "robot0_eye_in_hand"]
CAMERA_HEIGHT       = 84
CAMERA_WIDTH        = 84

# --- Training hyperparameters (BC) --------------------------------
BC_LEARNING_RATE    = 1e-4
BC_BATCH_SIZE       = 16
BC_NUM_EPOCHS       = 100
BC_SEQ_LENGTH       = 10
BC_ROLLOUT_FREQ     = 10    # evaluate every N epochs

# --- Training hyperparameters (Diffusion Policy) ------------------
DIFF_LEARNING_RATE  = 1e-4
DIFF_BATCH_SIZE     = 16
DIFF_NUM_EPOCHS     = 100
DIFF_ROLLOUT_FREQ   = 10

# --- Demo subsets for ablation study ------------------------------
DEMO_SUBSETS        = [5, 10, 20, 50]   # Section 6

print("Configuration loaded successfully.")
print(f"  Task    : {TASK_NAME}")
print(f"  Robot   : {ROBOT_NAME}")
print(f"  Base dir: {BASE_DIR}")

Configuration loaded successfully.
  Task    : PickPlaceCan
  Robot   : Panda
  Base dir: /tmp/rbe577_project4


---
## 1. Environment Setup

### 1.1 Install dependencies

Run the cell below **once** (or skip if your conda env is already set up).

In [ ]:
# Install robosuite & robomimic if not already present
import subprocess, sys

def run(cmd: str):
    """Run a shell command and stream output."""
    print(f">>> {cmd}")
    result = subprocess.run(cmd, shell=True, text=True,
                            capture_output=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    return result.returncode

# Uncomment the lines you need:
# run("pip install robosuite")
# run("pip install robomimic")

# Or install from source:
# run(f"cd {ROBOSUITE_ROOT} && pip install -r requirements.txt")
# run(f"cd {ROBOMIMIC_ROOT} && pip install -e .")

print("Done. Verify imports in next cell.")

### 1.2 Verify installations

In [ ]:
import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} "
      f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")

import robomimic
print(f"robomimic: OK  ({robomimic.__version__ if hasattr(robomimic,'__version__') else 'installed'})")

import robosuite
print(f"robosuite: OK  ({robosuite.__version__ if hasattr(robosuite,'__version__') else 'installed'})")

### 1.3 Create output directories

In [ ]:
import os

for d in [BASE_DIR, DEMO_DIR, BC_OUTPUT_DIR, DIFF_OUTPUT_DIR,
          f"{BASE_DIR}/tensorboard/bc",
          f"{BASE_DIR}/tensorboard/diffusion",
          f"{BASE_DIR}/videos",
          f"{BASE_DIR}/checkpoints",
          f"{BASE_DIR}/configs",
          f"{BASE_DIR}/subsets"]:
    os.makedirs(d, exist_ok=True)

print("Output directories created:")
for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

---
## 2. Collect Expert Demonstrations  *(40 pts)*

The cell below launches the robosuite keyboard teleoperation interface.  
**Do this in a terminal** – it requires a display / MuJoCo renderer.

Controls (keyboard):
| Key | Action |
|-----|--------|
| `W/S` | Forward / Back |
| `A/D` | Left / Right |
| `Z/X` | Up / Down |
| `G` | Toggle gripper |
| `Q` | Quit & save demo |

### 2.1 Launch teleoperation (run in terminal)

In [ ]:
# This cell prints the command to run in your terminal.
# DO NOT run this in a headless notebook – it needs a display.

collect_cmd = (
    f"python {ROBOSUITE_ROOT}/scripts/collect_human_demonstrations.py "
    f"--environment {TASK_NAME} "
    f"--robots {ROBOT_NAME} "
    f"--device keyboard "
    f"--renderer mujoco "
    f"--directory {DEMO_DIR}"
)

print("Run this command in a terminal with a display:")
print()
print(collect_cmd)
print()
print(f"Target: at least {MIN_DEMOS} successful demonstrations.")

### 2.2 Verify collected demonstrations

In [ ]:
import h5py, os

assert os.path.exists(DEMO_HDF5), (
    f"Demo file not found at {DEMO_HDF5}.\n"
    "Collect demonstrations first (Section 2.1).")

with h5py.File(DEMO_HDF5, "r") as f:
    demos = list(f["data"].keys())
    n_demos = len(demos)
    print(f"Dataset   : {DEMO_HDF5}")
    print(f"File size : {os.path.getsize(DEMO_HDF5) / 1e6:.1f} MB")
    print(f"Demos     : {n_demos}")
    if n_demos > 0:
        ep = f[f"data/{demos[0]}"]
        print(f"Keys (ep0): {list(ep.keys())}")
        print(f"Actions   : {ep['actions'].shape}")

assert n_demos >= MIN_DEMOS, (
    f"Only {n_demos} demos found – need at least {MIN_DEMOS}.")
print(f"\n✓ Sufficient demonstrations collected ({n_demos} >= {MIN_DEMOS}).")

---
## 3. Data Pipeline

### 3.1 Convert robosuite → robomimic format

In [ ]:
import subprocess

convert_cmd = (
    f"python {ROBOMIMIC_ROOT}/scripts/conversion/convert_robosuite.py "
    f"--dataset {DEMO_HDF5}"
)

print("Running conversion...")
print(convert_cmd)
result = subprocess.run(convert_cmd, shell=True, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Conversion failed.")
print("✓ Conversion complete.")

### 3.2 Create train / validation split

In [ ]:
split_cmd = (
    f"python {ROBOMIMIC_ROOT}/scripts/split_train_val.py "
    f"--dataset {DEMO_HDF5} "
    f"--ratio {TRAIN_VAL_RATIO}"
)

print("Creating train/val split...")
print(split_cmd)
result = subprocess.run(split_cmd, shell=True, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Split failed.")

# Verify split keys
with h5py.File(DEMO_HDF5, "r") as f:
    keys = list(f["mask"].keys()) if "mask" in f else []
    print(f"Split keys: {keys}")
    if "train" in keys and "valid" in keys:
        n_train = len(f["mask/train"])
        n_valid = len(f["mask/valid"])
        print(f"Train: {n_train}  |  Valid: {n_valid}")

print("✓ Train/val split created.")

### 3.3 Convert simulator states → image observations

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"   # change to 'osmesa' if egl fails

cam_str = " ".join(CAMERA_NAMES)

obs_cmd = (
    f"python {ROBOMIMIC_ROOT}/scripts/dataset_states_to_obs.py "
    f"--dataset {DEMO_HDF5} "
    f"--output_name {OBS_HDF5} "
    f"--done_mode 2 "
    f"--camera_names {cam_str} "
    f"--camera_height {CAMERA_HEIGHT} "
    f"--camera_width {CAMERA_WIDTH} "
    f"--compress"
)

print("Rendering image observations (may take several minutes)...")
print(obs_cmd)
result = subprocess.run(obs_cmd, shell=True, text=True, capture_output=True)
print(result.stdout[-3000:])  # tail of output
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("dataset_states_to_obs failed.")

size_mb = os.path.getsize(OBS_HDF5) / 1e6
print(f"\n✓ Observation dataset created: {OBS_HDF5} ({size_mb:.1f} MB)")

### 3.4 Inspect image observations

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

with h5py.File(OBS_HDF5, "r") as f:
    demo_keys = list(f["data"].keys())
    ep = f[f"data/{demo_keys[0]}"]
    obs = ep["obs"]

    print("Observation keys:", list(obs.keys()))

    fig, axes = plt.subplots(1, len(CAMERA_NAMES), figsize=(5 * len(CAMERA_NAMES), 4))
    if len(CAMERA_NAMES) == 1:
        axes = [axes]

    for ax, cam in zip(axes, CAMERA_NAMES):
        key = f"{cam}_image"
        if key in obs:
            img = obs[key][0]          # first timestep
            ax.imshow(img)
            ax.set_title(cam)
            ax.axis("off")
        else:
            ax.set_title(f"{cam} (not found)")

    plt.suptitle("Sample observations from demo 0, timestep 0", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/sample_observations.png", dpi=120)
    plt.show()
    print("✓ Observation images look good.")

---
## 4. Training Behavioral Cloning (BC)  *(20 pts)*

### 4.1 Generate BC config

In [ ]:
import json, copy, os

BC_CONFIG_PATH = f"{BASE_DIR}/configs/bc.json"

# Load the template
bc_template_path = f"{ROBOMIMIC_ROOT}/exps/templates/bc.json"
with open(bc_template_path, "r") as f:
    bc_cfg = json.load(f)

# ── Dataset paths & split keys ──────────────────────────────────
bc_cfg["train"]["data"]                         = OBS_HDF5
bc_cfg["train"]["hdf5_filter_key"]              = "train"
bc_cfg["train"]["hdf5_validation_filter_key"]   = "valid"

# ── Hyperparameters ─────────────────────────────────────────────
bc_cfg["train"]["batch_size"]                   = BC_BATCH_SIZE
bc_cfg["train"]["num_epochs"]                   = BC_NUM_EPOCHS
bc_cfg["train"]["seq_length"]                   = BC_SEQ_LENGTH
bc_cfg["train"]["rollout"]["rate"]              = BC_ROLLOUT_FREQ
bc_cfg["algo"]["optim_params"]["policy"]["learning_rate"]["initial"] = BC_LEARNING_RATE

# ── Logging ─────────────────────────────────────────────────────
bc_cfg["train"]["output_dir"]                   = BC_OUTPUT_DIR
bc_cfg["experiment"]["name"]                    = "bc_experiment"
bc_cfg["experiment"]["logging"]["log_tb"]       = True

with open(BC_CONFIG_PATH, "w") as f:
    json.dump(bc_cfg, f, indent=2)

print(f"BC config written to: {BC_CONFIG_PATH}")
print(f"  learning_rate : {BC_LEARNING_RATE}")
print(f"  batch_size    : {BC_BATCH_SIZE}")
print(f"  num_epochs    : {BC_NUM_EPOCHS}")
print(f"  seq_length    : {BC_SEQ_LENGTH}")
print(f"  rollout_freq  : {BC_ROLLOUT_FREQ}")

### 4.2 Train BC

In [ ]:
bc_train_cmd = (
    f"python {ROBOMIMIC_ROOT}/scripts/train.py "
    f"--config {BC_CONFIG_PATH} "
    f"--dataset {OBS_HDF5} "
    f"--name bc_experiment"
)

print("Starting BC training...")
print(bc_train_cmd)
print()
print("TensorBoard: run in a separate terminal:")
print(f"  tensorboard --logdir {BC_OUTPUT_DIR} --bind_all")
print()

result = subprocess.run(bc_train_cmd, shell=True, text=True, capture_output=True)
print(result.stdout[-5000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("BC training failed.")

print("✓ BC training complete.")

### 4.3 Locate BC checkpoint

In [ ]:
import glob

bc_ckpts = sorted(glob.glob(f"{BC_OUTPUT_DIR}/**/*.pth", recursive=True))
if bc_ckpts:
    BC_CHECKPOINT = bc_ckpts[-1]    # latest
    print(f"Found {len(bc_ckpts)} checkpoint(s). Using: {BC_CHECKPOINT}")
else:
    print("No checkpoint found yet – training may still be running.")
    BC_CHECKPOINT = None

---
## 5. Training Diffusion Policy  *(20 pts)*

### 5.1 Generate Diffusion Policy config

In [ ]:
DIFF_CONFIG_PATH = f"{BASE_DIR}/configs/diffusion_policy.json"

diff_template_path = f"{ROBOMIMIC_ROOT}/exps/templates/diffusion_policy.json"
with open(diff_template_path, "r") as f:
    diff_cfg = json.load(f)

# ── Dataset paths & split keys ──────────────────────────────────
diff_cfg["train"]["data"]                           = OBS_HDF5
diff_cfg["train"]["hdf5_filter_key"]                = "train"
diff_cfg["train"]["hdf5_validation_filter_key"]     = "valid"

# ── Hyperparameters ─────────────────────────────────────────────
diff_cfg["train"]["batch_size"]                     = DIFF_BATCH_SIZE
diff_cfg["train"]["num_epochs"]                     = DIFF_NUM_EPOCHS
diff_cfg["train"]["rollout"]["rate"]                = DIFF_ROLLOUT_FREQ
diff_cfg["algo"]["optim_params"]["policy"]["learning_rate"]["initial"] = DIFF_LEARNING_RATE

# ── Logging ─────────────────────────────────────────────────────
diff_cfg["train"]["output_dir"]                     = DIFF_OUTPUT_DIR
diff_cfg["experiment"]["name"]                      = "diffusion_experiment"
diff_cfg["experiment"]["logging"]["log_tb"]         = True

with open(DIFF_CONFIG_PATH, "w") as f:
    json.dump(diff_cfg, f, indent=2)

print(f"Diffusion config written to: {DIFF_CONFIG_PATH}")
print(f"  learning_rate : {DIFF_LEARNING_RATE}")
print(f"  batch_size    : {DIFF_BATCH_SIZE}")
print(f"  num_epochs    : {DIFF_NUM_EPOCHS}")
print(f"  rollout_freq  : {DIFF_ROLLOUT_FREQ}")

### 5.2 Train Diffusion Policy

In [ ]:
diff_train_cmd = (
    f"python {ROBOMIMIC_ROOT}/scripts/train.py "
    f"--config {DIFF_CONFIG_PATH} "
    f"--dataset {OBS_HDF5} "
    f"--name diffusion_experiment"
)

print("Starting Diffusion Policy training...")
print(diff_train_cmd)
print()
print("TensorBoard: run in a separate terminal:")
print(f"  tensorboard --logdir {DIFF_OUTPUT_DIR} --bind_all")
print()

result = subprocess.run(diff_train_cmd, shell=True, text=True, capture_output=True)
print(result.stdout[-5000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("Diffusion training failed.")

print("✓ Diffusion Policy training complete.")

### 5.3 Locate Diffusion checkpoint

In [ ]:
diff_ckpts = sorted(glob.glob(f"{DIFF_OUTPUT_DIR}/**/*.pth", recursive=True))
if diff_ckpts:
    DIFF_CHECKPOINT = diff_ckpts[-1]
    print(f"Found {len(diff_ckpts)} checkpoint(s). Using: {DIFF_CHECKPOINT}")
else:
    print("No checkpoint found yet – training may still be running.")
    DIFF_CHECKPOINT = None

---
## 6. Ablation Study: Success Rate vs. Number of Demonstrations  *(Discussion)*

Train both methods on subsets of [5, 10, 20, 50] demos and record the best rollout success rate.

### 6.1 Create demo subsets

In [ ]:
import h5py, os, shutil

def create_demo_subset(src_hdf5: str, n_demos: int, out_dir: str) -> str:
    """Copy the first n_demos demonstrations into a new HDF5 file."""
    out_path = os.path.join(out_dir, f"obs_{n_demos}_demos.hdf5")
    if os.path.exists(out_path):
        print(f"  Subset already exists: {out_path}")
        return out_path

    with h5py.File(src_hdf5, "r") as src, h5py.File(out_path, "w") as dst:
        # Copy top-level attrs
        for k, v in src.attrs.items():
            dst.attrs[k] = v

        demo_keys = list(src["data"].keys())[:n_demos]
        src.copy("data", dst, name="data")

        # Remove demos beyond n_demos
        all_keys = list(dst["data"].keys())
        for k in all_keys:
            if k not in demo_keys:
                del dst[f"data/{k}"]

        # Rebuild mask
        if "mask" in src:
            pass  # will re-split below

    print(f"  Created subset ({n_demos} demos): {out_path}")
    return out_path


subset_dir = f"{BASE_DIR}/subsets"
subset_paths = {}

for n in DEMO_SUBSETS:
    print(f"Creating subset: {n} demos")
    path = create_demo_subset(OBS_HDF5, n, subset_dir)
    subset_paths[n] = path

print("\n✓ All subsets created.")
for n, p in subset_paths.items():
    print(f"  {n:2d} demos → {p}")

### 6.2 Re-split each subset

In [ ]:
for n, path in subset_paths.items():
    split_cmd = (
        f"python {ROBOMIMIC_ROOT}/scripts/split_train_val.py "
        f"--dataset {path} "
        f"--ratio {TRAIN_VAL_RATIO}"
    )
    result = subprocess.run(split_cmd, shell=True, text=True, capture_output=True)
    if result.returncode == 0:
        print(f"  Split OK: {n} demos")
    else:
        print(f"  Split FAILED ({n} demos): {result.stderr[:200]}")

### 6.3 Train BC on each subset

In [ ]:
import re

def extract_best_success_rate(log_output: str) -> float:
    """Parse the highest Rollout_Success_Rate from training stdout."""
    pattern = r"Rollout_Success_Rate.*?([0-9]+\.?[0-9]*)"
    matches = re.findall(pattern, log_output)
    return max((float(m) for m in matches), default=0.0)


bc_subset_results = {}   # {n_demos: success_rate}

for n, path in subset_paths.items():
    print(f"\n─── BC training on {n} demos ───")

    # Write a config for this subset
    cfg = copy.deepcopy(bc_cfg)
    cfg["train"]["data"]      = path
    cfg["experiment"]["name"] = f"bc_{n}demos"
    cfg["train"]["output_dir"]= f"{BASE_DIR}/bc_{n}demos"
    cfg_path = f"{BASE_DIR}/configs/bc_{n}demos.json"
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

    cmd = (
        f"python {ROBOMIMIC_ROOT}/scripts/train.py "
        f"--config {cfg_path} --dataset {path}"
    )
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    rate = extract_best_success_rate(result.stdout)
    bc_subset_results[n] = rate
    print(f"  Best success rate ({n} demos): {rate:.3f}")

print("\nBC subset results:", bc_subset_results)

### 6.4 Train Diffusion Policy on each subset

In [ ]:
diff_subset_results = {}   # {n_demos: success_rate}

for n, path in subset_paths.items():
    print(f"\n─── Diffusion training on {n} demos ───")

    cfg = copy.deepcopy(diff_cfg)
    cfg["train"]["data"]      = path
    cfg["experiment"]["name"] = f"diffusion_{n}demos"
    cfg["train"]["output_dir"]= f"{BASE_DIR}/diffusion_{n}demos"
    cfg_path = f"{BASE_DIR}/configs/diffusion_{n}demos.json"
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

    cmd = (
        f"python {ROBOMIMIC_ROOT}/scripts/train.py "
        f"--config {cfg_path} --dataset {path}"
    )
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    rate = extract_best_success_rate(result.stdout)
    diff_subset_results[n] = rate
    print(f"  Best success rate ({n} demos): {rate:.3f}")

print("\nDiffusion subset results:", diff_subset_results)

### 6.5 Plot: Success Rate vs. Number of Demonstrations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

demo_counts = sorted(DEMO_SUBSETS)

bc_rates   = [bc_subset_results.get(n, 0.0)   for n in demo_counts]
diff_rates = [diff_subset_results.get(n, 0.0) for n in demo_counts]

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(demo_counts, bc_rates,   marker="o", linewidth=2,
        label="Behavioral Cloning",  color="steelblue")
ax.plot(demo_counts, diff_rates, marker="s", linewidth=2,
        label="Diffusion Policy",    color="coral", linestyle="--")

ax.set_xlabel("Number of Demonstrations", fontsize=12)
ax.set_ylabel("Best Rollout Success Rate", fontsize=12)
ax.set_title(f"Success Rate vs. Number of Demos\n({TASK_NAME}, {ROBOT_NAME})",
             fontsize=13)
ax.set_xticks(demo_counts)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = f"{BASE_DIR}/success_rate_vs_demos.png"
plt.savefig(out_path, dpi=150)
plt.show()
print(f"Saved: {out_path}")

---
## 7. Hyperparameter Analysis  *(Discussion)*

Compare two sets of hyperparameters for each method and plot training/validation loss.

### 7.1 BC: Two hyperparameter configs

In [ ]:
BC_HPARAM_CONFIGS = [
    {"name": "bc_lr1e-4_bs16",  "lr": 1e-4, "bs": 16,  "epochs": 50},
    {"name": "bc_lr1e-3_bs32",  "lr": 1e-3, "bs": 32,  "epochs": 50},
]

bc_hparam_logs = {}   # name -> {train_loss, val_loss}

for hp in BC_HPARAM_CONFIGS:
    print(f"\nTraining: {hp['name']}")

    cfg = copy.deepcopy(bc_cfg)
    cfg["train"]["data"]       = OBS_HDF5
    cfg["train"]["batch_size"] = hp["bs"]
    cfg["train"]["num_epochs"] = hp["epochs"]
    cfg["algo"]["optim_params"]["policy"]["learning_rate"]["initial"] = hp["lr"]
    cfg["experiment"]["name"]  = hp["name"]
    cfg["train"]["output_dir"] = f"{BASE_DIR}/{hp['name']}"

    cfg_path = f"{BASE_DIR}/configs/{hp['name']}.json"
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

    cmd = (
        f"python {ROBOMIMIC_ROOT}/scripts/train.py "
        f"--config {cfg_path} --dataset {OBS_HDF5}"
    )
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)

    # Parse train/val loss from stdout
    train_losses = [float(x) for x in re.findall(r"Train_Loss.*?([0-9]+\.[0-9]+)", result.stdout)]
    val_losses   = [float(x) for x in re.findall(r"Valid_Loss.*?([0-9]+\.[0-9]+)", result.stdout)]
    bc_hparam_logs[hp["name"]] = {"train": train_losses, "val": val_losses}
    print(f"  Epochs recorded: train={len(train_losses)}, val={len(val_losses)}")

print("\n✓ BC hyperparameter runs complete.")

### 7.2 Plot BC hyperparameter comparison

In [ ]:
colors = ["steelblue", "coral", "green", "purple"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for i, hp in enumerate(BC_HPARAM_CONFIGS):
    name = hp["name"]
    data = bc_hparam_logs.get(name, {})
    label = f"lr={hp['lr']}, bs={hp['bs']}"

    if data.get("train"):
        ax1.plot(data["train"], label=label, color=colors[i])
    if data.get("val"):
        ax2.plot(data["val"],   label=label, color=colors[i], linestyle="--")

ax1.set_title("BC Training Loss", fontsize=12)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_title("BC Validation Loss", fontsize=12)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("BC Hyperparameter Comparison", fontsize=13)
plt.tight_layout()
out = f"{BASE_DIR}/bc_hyperparam_comparison.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

### 7.3 Diffusion Policy: Two hyperparameter configs

In [ ]:
DIFF_HPARAM_CONFIGS = [
    {"name": "diff_lr1e-4_bs16", "lr": 1e-4, "bs": 16, "epochs": 50},
    {"name": "diff_lr1e-3_bs32", "lr": 1e-3, "bs": 32, "epochs": 50},
]

diff_hparam_logs = {}

for hp in DIFF_HPARAM_CONFIGS:
    print(f"\nTraining: {hp['name']}")

    cfg = copy.deepcopy(diff_cfg)
    cfg["train"]["data"]       = OBS_HDF5
    cfg["train"]["batch_size"] = hp["bs"]
    cfg["train"]["num_epochs"] = hp["epochs"]
    cfg["algo"]["optim_params"]["policy"]["learning_rate"]["initial"] = hp["lr"]
    cfg["experiment"]["name"]  = hp["name"]
    cfg["train"]["output_dir"] = f"{BASE_DIR}/{hp['name']}"

    cfg_path = f"{BASE_DIR}/configs/{hp['name']}.json"
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

    cmd = (
        f"python {ROBOMIMIC_ROOT}/scripts/train.py "
        f"--config {cfg_path} --dataset {OBS_HDF5}"
    )
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)

    train_losses = [float(x) for x in re.findall(r"Train_Loss.*?([0-9]+\.[0-9]+)", result.stdout)]
    val_losses   = [float(x) for x in re.findall(r"Valid_Loss.*?([0-9]+\.[0-9]+)", result.stdout)]
    diff_hparam_logs[hp["name"]] = {"train": train_losses, "val": val_losses}
    print(f"  Recorded: train={len(train_losses)}, val={len(val_losses)}")

print("✓ Diffusion hyperparameter runs complete.")

### 7.4 Plot Diffusion hyperparameter comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for i, hp in enumerate(DIFF_HPARAM_CONFIGS):
    name  = hp["name"]
    data  = diff_hparam_logs.get(name, {})
    label = f"lr={hp['lr']}, bs={hp['bs']}"

    if data.get("train"):
        ax1.plot(data["train"], label=label, color=colors[i])
    if data.get("val"):
        ax2.plot(data["val"],   label=label, color=colors[i], linestyle="--")

ax1.set_title("Diffusion Training Loss", fontsize=12)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_title("Diffusion Validation Loss", fontsize=12)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Diffusion Policy Hyperparameter Comparison", fontsize=13)
plt.tight_layout()
out = f"{BASE_DIR}/diffusion_hyperparam_comparison.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

---
## 8. Evaluation & Rollout Videos

### 8.1 Run rollout evaluation for BC

In [ ]:
BC_VIDEO_PATH   = f"{BASE_DIR}/videos/bc_rollout.mp4"

if BC_CHECKPOINT:
    bc_eval_cmd = (
        f"python {ROBOMIMIC_ROOT}/scripts/run_trained_agent.py "
        f"--agent {BC_CHECKPOINT} "
        f"--n_rollouts 10 "
        f"--horizon 400 "
        f"--seed 0 "
        f"--video_path {BC_VIDEO_PATH}"
    )
    print("Evaluating BC policy...")
    print(bc_eval_cmd)
    result = subprocess.run(bc_eval_cmd, shell=True,
                            text=True, capture_output=True)
    print(result.stdout[-3000:])

    # Parse success rate from output
    sr_match = re.search(r"Success_Rate.*?([0-9]+\.?[0-9]*)", result.stdout)
    BC_FINAL_SUCCESS_RATE = float(sr_match.group(1)) if sr_match else None
    print(f"\nBC Final Success Rate: {BC_FINAL_SUCCESS_RATE}")
else:
    print("No BC checkpoint found – skipping evaluation.")
    BC_FINAL_SUCCESS_RATE = None

### 8.2 Run rollout evaluation for Diffusion Policy

In [ ]:
DIFF_VIDEO_PATH = f"{BASE_DIR}/videos/diffusion_rollout.mp4"

if DIFF_CHECKPOINT:
    diff_eval_cmd = (
        f"python {ROBOMIMIC_ROOT}/scripts/run_trained_agent.py "
        f"--agent {DIFF_CHECKPOINT} "
        f"--n_rollouts 10 "
        f"--horizon 400 "
        f"--seed 0 "
        f"--video_path {DIFF_VIDEO_PATH}"
    )
    print("Evaluating Diffusion Policy...")
    print(diff_eval_cmd)
    result = subprocess.run(diff_eval_cmd, shell=True,
                            text=True, capture_output=True)
    print(result.stdout[-3000:])

    sr_match = re.search(r"Success_Rate.*?([0-9]+\.?[0-9]*)", result.stdout)
    DIFF_FINAL_SUCCESS_RATE = float(sr_match.group(1)) if sr_match else None
    print(f"\nDiffusion Final Success Rate: {DIFF_FINAL_SUCCESS_RATE}")
else:
    print("No Diffusion checkpoint found – skipping evaluation.")
    DIFF_FINAL_SUCCESS_RATE = None

### 8.3 Final performance comparison bar chart

In [ ]:
methods = ["Behavioral Cloning", "Diffusion Policy"]
rates   = [BC_FINAL_SUCCESS_RATE or 0.0,
           DIFF_FINAL_SUCCESS_RATE or 0.0]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(methods, rates, color=["steelblue", "coral"],
              edgecolor="black", width=0.5)

for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{rate:.2f}", ha="center", va="bottom", fontsize=12)

ax.set_ylim(0, 1.15)
ax.set_ylabel("Success Rate (10 rollouts)", fontsize=12)
ax.set_title(f"Final Evaluation – {TASK_NAME}", fontsize=13)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
out = f"{BASE_DIR}/final_comparison.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

---
## 9. Package Submission Files

### 9.1 Copy checkpoints & configs to organised output

In [ ]:
import shutil, os

# Checkpoints
if BC_CHECKPOINT and os.path.exists(BC_CHECKPOINT):
    dst = f"{BASE_DIR}/checkpoints/bc_model.pth"
    shutil.copy2(BC_CHECKPOINT, dst)
    print(f"Copied BC checkpoint → {dst}")

if DIFF_CHECKPOINT and os.path.exists(DIFF_CHECKPOINT):
    dst = f"{BASE_DIR}/checkpoints/diffusion_model.pth"
    shutil.copy2(DIFF_CHECKPOINT, dst)
    print(f"Copied Diffusion checkpoint → {dst}")

# Configs
if os.path.exists(BC_CONFIG_PATH):
    shutil.copy2(BC_CONFIG_PATH,   f"{BASE_DIR}/configs/bc.json")
if os.path.exists(DIFF_CONFIG_PATH):
    shutil.copy2(DIFF_CONFIG_PATH, f"{BASE_DIR}/configs/diffusion_policy.json")

print("✓ Files organised.")

### 9.2 Create submission ZIP

In [ ]:
import zipfile, os

YOUR_NAME   = "YourName"    # <── replace before running
ZIP_PATH    = f"/tmp/project_{YOUR_NAME}.zip"

INCLUDE_PATTERNS = [
    "configs/bc.json",
    "configs/diffusion_policy.json",
    "checkpoints/bc_model.pth",
    "checkpoints/diffusion_model.pth",
    "videos/bc_rollout.mp4",
    "videos/diffusion_rollout.mp4",
    "success_rate_vs_demos.png",
    "bc_hyperparam_comparison.png",
    "diffusion_hyperparam_comparison.png",
    "final_comparison.png",
    "sample_observations.png",
]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for rel in INCLUDE_PATTERNS:
        full = os.path.join(BASE_DIR, rel)
        if os.path.exists(full):
            zf.write(full, arcname=rel)
            print(f"  Added: {rel}")
        else:
            print(f"  MISSING: {rel}")

    # Include this notebook
    nb_path = os.path.abspath("project4_imitation_learning.ipynb")
    if os.path.exists(nb_path):
        zf.write(nb_path, arcname="scripts/project4_imitation_learning.ipynb")

print(f"\n✓ ZIP created: {ZIP_PATH}")
print(f"   Size: {os.path.getsize(ZIP_PATH) / 1e6:.1f} MB")

---
## 10. Summary & Discussion Notes

Use this cell to record your findings for the written report.

In [ ]:
print("=" * 60)
print("PROJECT 4 SUMMARY")
print("=" * 60)
print(f"Task            : {TASK_NAME}")
print(f"Robot           : {ROBOT_NAME}")
print(f"Cameras         : {CAMERA_NAMES}")
print()

try:
    with h5py.File(DEMO_HDF5, "r") as f:
        print(f"Total demos     : {len(f['data'])}")
except:
    print("Total demos     : (demo file not found)")

print(f"Train/val split : {1 - TRAIN_VAL_RATIO:.0%} / {TRAIN_VAL_RATIO:.0%}")
print()
print("── Behavioral Cloning ──")
print(f"  LR={BC_LEARNING_RATE}, BS={BC_BATCH_SIZE}, "
      f"epochs={BC_NUM_EPOCHS}, seq={BC_SEQ_LENGTH}")
print(f"  Final success rate : {BC_FINAL_SUCCESS_RATE}")
print()
print("── Diffusion Policy ──")
print(f"  LR={DIFF_LEARNING_RATE}, BS={DIFF_BATCH_SIZE}, "
      f"epochs={DIFF_NUM_EPOCHS}")
print(f"  Final success rate : {DIFF_FINAL_SUCCESS_RATE}")
print()
print("── Demo-count ablation ──")
for n in DEMO_SUBSETS:
    bc_r   = bc_subset_results.get(n, "N/A")
    diff_r = diff_subset_results.get(n, "N/A")
    print(f"  {n:2d} demos  →  BC={bc_r}   Diff={diff_r}")
print("=" * 60)

---
## Appendix: TensorBoard Quick Reference

```bash
# BC
tensorboard --logdir /tmp/rbe577_project4/bc_output --bind_all

# Diffusion
tensorboard --logdir /tmp/rbe577_project4/diffusion_output --bind_all

# Both at once
tensorboard --logdir bc:/tmp/rbe577_project4/bc_output,\
diffusion:/tmp/rbe577_project4/diffusion_output --bind_all
```

## Appendix: MuJoCo Rendering Fixes

```bash
export MUJOCO_GL=egl     # try first
export MUJOCO_GL=osmesa  # fallback if egl fails
```

## Appendix: Debug Mode

Add `--debug` to any `train.py` call for a short sanity-check run:

```bash
python robomimic/scripts/train.py --config bc.json --dataset obs.hdf5 --debug
```